# Notebook 23 — Drift Adaptation

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 19 detected stale-context risk.

Notebook 23 asks:

> Can the system revise after context shifts?

The central pattern is:

\[
\text{context shift}
\rightarrow
\text{stale risk spike}
\rightarrow
\text{revision}
\rightarrow
\text{recovery}
\]

Stale context is not failure by itself. Failure occurs when stale context is not revised.

Outputs:

- `results/23_drift_candidates.csv`
- `results/23_drift_recovery.csv`
- `results/23_revision_scores.csv`
- `results/23_drift_adaptation_summary.json`
- `figures/23_drift_candidate_timeline.png`
- `figures/23_recovery_after_drift.png`
- `figures/23_adaptation_lag.png`
- `figures/23_revision_success_map.png`
- `results/23_drift_adaptation_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain

print("Imports complete.")

## 2. Load Prior Artifacts

Notebook 23 prefers Notebook 19 stale-context events:

```text
results/19_stale_context_events.csv
```

It also uses optional stale-vs-plasticity and boundary recovery outputs.

Fallback order for trajectory:

1. `results/19_stale_context_events.csv`
2. `results/17_plasticity_states.csv`
3. `results/13_context_stability.csv`
4. `results/11_state_vs_stateless_instances.csv`
5. `results/01_gain_signal_trajectory.csv`
6. `results/00_context_gain.csv`
7. synthetic toy trajectory

In [ ]:
trajectory_paths = [
    RESULTS_DIR / "19_stale_context_events.csv",
    RESULTS_DIR / "17_plasticity_states.csv",
    RESULTS_DIR / "13_context_stability.csv",
    RESULTS_DIR / "11_state_vs_stateless_instances.csv",
    RESULTS_DIR / "01_gain_signal_trajectory.csv",
    RESULTS_DIR / "00_context_gain.csv",
]

source_file = None
df = None

for path in trajectory_paths:
    if path.exists():
        df = pd.read_csv(path)
        source_file = path
        print("Loaded:", path)
        break

if df is None:
    print("No prior trajectory found. Creating fallback toy trajectory.")
    df = pd.DataFrame({
        "instance": np.arange(1, 13),
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })

if "gain" not in df.columns:
    if "state_advantage" in df.columns:
        df["gain"] = df["state_advantage"]
    else:
        df["gain"] = compute_gain(
            df["stateful_reward"].tolist(),
            df["stateless_reward"].tolist(),
        )

if "gain_delta" not in df.columns:
    df["gain_delta"] = df["gain"].diff().fillna(0.0)

if "cumulative_gain" not in df.columns:
    df["cumulative_gain"] = df["gain"].cumsum()

df = df.sort_values("instance").reset_index(drop=True)

if "stale_risk_score" not in df.columns:
    df["stale_risk_score"] = 0.0

if "stale_risk_level" not in df.columns:
    df["stale_risk_level"] = "none"

stale_vs_plasticity_path = RESULTS_DIR / "19_stale_vs_plasticity.csv"
if stale_vs_plasticity_path.exists():
    stale_vs_plasticity = pd.read_csv(stale_vs_plasticity_path)
    print("Loaded:", stale_vs_plasticity_path)
else:
    stale_vs_plasticity = None

boundary_recovery_path = RESULTS_DIR / "17_boundary_recovery.csv"
if boundary_recovery_path.exists():
    boundary_recovery = pd.read_csv(boundary_recovery_path)
    print("Loaded:", boundary_recovery_path)
else:
    boundary_recovery = None

df

## 3. Define Drift Adaptation

A **drift candidate** is a context transition with stale-risk evidence.

Operationally:

\[
\text{drift candidate}
=
\text{boundary}
\land
\text{stale risk score} > \tau
\]

Adaptation is measured by recovery after the drift event:

\[
\text{recovery after drift}
=
\max(g_{future})
-
g_{drift}
\]

A successful revision means the system recovers after the risk spike.

## 4. Variant Boundaries and Drift Candidates

In [ ]:
variant_order = (
    df[["variant", "instance"]]
    .drop_duplicates("variant")
    .sort_values("instance")
)

boundary_instances = variant_order["instance"].tolist()

if "boundary_flag" not in df.columns:
    df["boundary_flag"] = df["instance"].isin(boundary_instances)

risk_threshold = 1.0

df["drift_candidate"] = (
    df["boundary_flag"].astype(bool)
    & (df["stale_risk_score"] >= risk_threshold)
)

# Also flag strong non-boundary drift candidates from large gain drops.
if "large_gain_drop_flag" in df.columns:
    df["non_boundary_drift_candidate"] = (
        (~df["boundary_flag"].astype(bool))
        & df["large_gain_drop_flag"].astype(bool)
    )
else:
    df["non_boundary_drift_candidate"] = (
        (~df["boundary_flag"].astype(bool))
        & (df["gain_delta"] <= -0.08)
    )

drift_candidates = df[
    df["drift_candidate"] | df["non_boundary_drift_candidate"]
].copy()

if drift_candidates.empty:
    # Keep the strongest risk event so downstream cells remain informative.
    strongest_idx = df["stale_risk_score"].idxmax()
    drift_candidates = df.loc[[strongest_idx]].copy()
    drift_candidates["fallback_candidate"] = True
else:
    drift_candidates["fallback_candidate"] = False

drift_candidates[[
    "instance",
    "variant",
    "gain",
    "gain_delta",
    "stale_risk_score",
    "stale_risk_level",
    "boundary_flag",
    "drift_candidate",
    "non_boundary_drift_candidate",
    "fallback_candidate",
]]

## 5. Recovery After Drift

For each drift candidate:

- record gain at drift,
- find peak future gain in the same variant,
- compute recovery,
- measure whether the system exceeds pre-drift gain.

In [ ]:
recovery_rows = []

for _, event in drift_candidates.iterrows():
    variant = event["variant"]
    instance = int(event["instance"])
    gain_at_drift = float(event["gain"])

    same_variant_future = df[
        (df["variant"] == variant)
        & (df["instance"] >= instance)
    ].copy()

    if same_variant_future.empty:
        peak_future_gain = gain_at_drift
        peak_instance = instance
        final_gain = gain_at_drift
    else:
        peak_idx = same_variant_future["gain"].idxmax()
        peak_future_gain = float(df.loc[peak_idx, "gain"])
        peak_instance = int(df.loc[peak_idx, "instance"])
        final_gain = float(same_variant_future["gain"].iloc[-1])

    prior_row = df[df["instance"] < instance].tail(1)
    if prior_row.empty:
        pre_drift_gain = np.nan
    else:
        pre_drift_gain = float(prior_row["gain"].iloc[0])

    recovery_gain = peak_future_gain - gain_at_drift
    final_recovery = final_gain - gain_at_drift

    if np.isnan(pre_drift_gain):
        exceeded_pre_drift = False
        adaptation_lag = np.nan
    else:
        future_exceed = same_variant_future[
            same_variant_future["gain"] >= pre_drift_gain
        ]

        if future_exceed.empty:
            exceeded_pre_drift = False
            adaptation_lag = np.nan
        else:
            exceeded_pre_drift = True
            adaptation_lag = int(future_exceed["instance"].iloc[0] - instance)

    if recovery_gain > 0.10 and exceeded_pre_drift:
        revision_label = "successful_revision"
    elif recovery_gain > 0:
        revision_label = "partial_recovery"
    elif recovery_gain == 0:
        revision_label = "no_recovery"
    else:
        revision_label = "continued_decline"

    recovery_rows.append({
        "drift_instance": instance,
        "variant": variant,
        "gain_at_drift": gain_at_drift,
        "pre_drift_gain": pre_drift_gain,
        "peak_future_gain": peak_future_gain,
        "peak_future_instance": peak_instance,
        "final_gain_in_variant": final_gain,
        "recovery_gain": float(recovery_gain),
        "final_recovery": float(final_recovery),
        "exceeded_pre_drift_gain": bool(exceeded_pre_drift),
        "adaptation_lag_instances": adaptation_lag,
        "revision_label": revision_label,
        "stale_risk_score": float(event["stale_risk_score"]),
    })

drift_recovery = pd.DataFrame(recovery_rows)
drift_recovery

## 6. Revision Success Score

Revision success rewards recovery and penalizes lag.

A simple score:

\[
\text{revision score}
=
\text{recovery gain}
-
0.02 \times \text{adaptation lag}
\]

This is intentionally lightweight and interpretable.

In [ ]:
revision_scores = drift_recovery.copy()

revision_scores["lag_penalty"] = revision_scores["adaptation_lag_instances"].fillna(5) * 0.02
revision_scores["revision_success_score"] = (
    revision_scores["recovery_gain"]
    - revision_scores["lag_penalty"]
)

def classify_revision(score):
    if score >= 0.10:
        return "strong_revision"
    if score > 0:
        return "weak_revision"
    return "failed_revision"

revision_scores["revision_success_class"] = revision_scores[
    "revision_success_score"
].apply(classify_revision)

revision_scores

## 7. Figure — Drift Candidate Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    df["instance"],
    df["stale_risk_score"],
    label="stale risk",
)

candidate_instances = drift_candidates["instance"].tolist()

for inst in candidate_instances:
    ax.axvline(inst, linestyle="--", linewidth=1.5)

for boundary in boundary_instances[1:]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 23: Drift Candidate Timeline")
ax.set_xlabel("Instance")
ax.set_ylabel("Stale Risk Score")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

drift_candidate_timeline_path = FIGURES_DIR / "23_drift_candidate_timeline.png"
fig.tight_layout()
fig.savefig(drift_candidate_timeline_path, dpi=160)

drift_candidate_timeline_path

## 8. Figure — Recovery After Drift

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

labels = [
    f"{row['variant']}\ninst {int(row['drift_instance'])}"
    for _, row in drift_recovery.iterrows()
]

ax.bar(
    labels,
    drift_recovery["recovery_gain"],
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 23: Recovery After Drift")
ax.set_xlabel("Drift Candidate")
ax.set_ylabel("Recovery Gain")
ax.grid(True, axis="y", alpha=0.3)

recovery_after_drift_path = FIGURES_DIR / "23_recovery_after_drift.png"
fig.tight_layout()
fig.savefig(recovery_after_drift_path, dpi=160)

recovery_after_drift_path

## 9. Figure — Adaptation Lag

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

lag_values = drift_recovery["adaptation_lag_instances"].fillna(-1)

ax.bar(
    labels,
    lag_values,
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 23: Adaptation Lag")
ax.set_xlabel("Drift Candidate")
ax.set_ylabel("Instances Until Pre-Drift Gain Recovered (-1 = not recovered)")
ax.grid(True, axis="y", alpha=0.3)

adaptation_lag_path = FIGURES_DIR / "23_adaptation_lag.png"
fig.tight_layout()
fig.savefig(adaptation_lag_path, dpi=160)

adaptation_lag_path

## 10. Figure — Revision Success Map

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    revision_scores["stale_risk_score"],
    revision_scores["revision_success_score"],
    s=140,
)

for _, row in revision_scores.iterrows():
    ax.annotate(
        f"{row['variant']}:{int(row['drift_instance'])}",
        (row["stale_risk_score"], row["revision_success_score"]),
        textcoords="offset points",
        xytext=(8, 8),
    )

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 23: Revision Success Map")
ax.set_xlabel("Stale Risk Score")
ax.set_ylabel("Revision Success Score")
ax.grid(True, alpha=0.3)

revision_success_map_path = FIGURES_DIR / "23_revision_success_map.png"
fig.tight_layout()
fig.savefig(revision_success_map_path, dpi=160)

revision_success_map_path

## 11. Summary

In [ ]:
best_revision = revision_scores.loc[
    revision_scores["revision_success_score"].idxmax()
].to_dict()

worst_revision = revision_scores.loc[
    revision_scores["revision_success_score"].idxmin()
].to_dict()

summary = {
    "total_instances": int(len(df)),
    "variants": df["variant"].drop_duplicates().tolist(),
    "trajectory_source": str(source_file) if source_file is not None else "fallback",
    "drift_candidate_count": int(len(drift_candidates)),
    "risk_threshold": risk_threshold,
    "mean_recovery_gain": float(drift_recovery["recovery_gain"].mean()),
    "max_recovery_gain": float(drift_recovery["recovery_gain"].max()),
    "mean_revision_success_score": float(revision_scores["revision_success_score"].mean()),
    "best_revision_variant": str(best_revision["variant"]),
    "best_revision_instance": int(best_revision["drift_instance"]),
    "best_revision_success_score": float(best_revision["revision_success_score"]),
    "worst_revision_variant": str(worst_revision["variant"]),
    "worst_revision_instance": int(worst_revision["drift_instance"]),
    "worst_revision_success_score": float(worst_revision["revision_success_score"]),
    "revision_success_classes": {
        label: int(count)
        for label, count in revision_scores["revision_success_class"].value_counts().items()
    },
}

summary

## 12. Export Artifacts

In [ ]:
drift_candidates_csv = RESULTS_DIR / "23_drift_candidates.csv"
drift_recovery_csv = RESULTS_DIR / "23_drift_recovery.csv"
revision_scores_csv = RESULTS_DIR / "23_revision_scores.csv"
summary_path = RESULTS_DIR / "23_drift_adaptation_summary.json"
zip_path = RESULTS_DIR / "23_drift_adaptation_artifacts.zip"

drift_candidates.to_csv(drift_candidates_csv, index=False)
drift_recovery.to_csv(drift_recovery_csv, index=False)
revision_scores.to_csv(revision_scores_csv, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "23_drift_adaptation.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(summary_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        drift_candidates_csv,
        drift_recovery_csv,
        revision_scores_csv,
        summary_path,
        drift_candidate_timeline_path,
        recovery_after_drift_path,
        adaptation_lag_path,
        revision_success_map_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    drift_candidates_csv,
    drift_recovery_csv,
    revision_scores_csv,
    summary_path,
    drift_candidate_timeline_path,
    recovery_after_drift_path,
    adaptation_lag_path,
    revision_success_map_path,
    zip_path,
]:
    print("-", path)

## 13. Download Artifacts

In Colab, this downloads the zip.

Locally, the archive is saved under:

```text
rml/results/23_drift_adaptation_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 14. Notebook 23 Claim

Stale context is not failure by itself.

Failure occurs when stale context is not revised.

\[
\text{stale risk}
\rightarrow
\text{drift detection}
\rightarrow
\text{revision}
\rightarrow
\text{recovery}
\]

In RML terms:

> continual learning requires not only retaining useful structure, but revising structure when context changes.

Notebook 29 will close the sequence with a failure-mode taxonomy.